In [3]:
import polars as pl
import os
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive


In [4]:
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/2019-Oct.csv'
print(f"Размер файла: {os.path.getsize(file_path) / (1024**3):.2f} GB")

Mounted at /content/drive
Размер файла: 5.28 GB


In [5]:
#Polars для предварительного чтения и фильтрации

lf = pl.scan_csv(file_path, ignore_errors=True)

session_df = (
    lf
    .select([
        "event_time", "event_type", "category_code", "brand",
        "price", "user_id", "user_session"
    ])

    .filter(
        (pl.col("price") > 0) &
        (pl.col("event_type").is_in(["view", "cart", "purchase"])) &
        (pl.col("brand").is_not_null() | pl.col("category_code").is_not_null()) &
        (pl.col("event_time") >= "2019-10-15") &
        (pl.col("event_time") < "2019-10-29")
    )
    # Парсинг даты
    .with_columns(
        pl.col("event_time")
          .str.slice(0, 19)
          .str.to_datetime(format="%Y-%m-%d %H:%M:%S")
          .alias("event_dt")
    )
    # Признаки для анализа
    .with_columns(
        (pl.col("event_type") == "view").cast(pl.Int8).alias("is_view"),
        (pl.col("event_type") == "cart").cast(pl.Int8).alias("is_cart"),
        (pl.col("event_type") == "purchase").cast(pl.Int8).alias("is_purchase"),
        pl.col("event_dt").dt.hour().alias("hour"),
        pl.col("event_dt").dt.weekday().alias("weekday"),
        (pl.col("event_dt").dt.weekday() >= 5).cast(pl.Int8).alias("is_weekend"),
        pl.col("category_code")
          .str.split(".")
          .list.get(0)
          .alias("main_category")
    )
    # Агрегация по сессиям
    .group_by(["user_session", "user_id"])
    .agg(
        pl.col("weekday").first().alias("weekday"),
        pl.col("is_weekend").first().alias("is_weekend"),
        pl.col("hour").min().alias("start_hour"),
        pl.len().alias("events_count"),
        pl.col("is_view").sum().alias("views"),
        pl.col("is_cart").sum().alias("carts"),
        pl.col("is_purchase").sum().alias("purchases"),
        (pl.col("is_purchase").sum() > 0).cast(pl.Int8).alias("made_purchase"),
        (pl.col("is_cart").sum() > 0).cast(pl.Int8).alias("added_to_cart"),
        pl.col("price").mean().alias("avg_price"),
        pl.col("price").max().alias("max_price"),
        pl.col("main_category")
          .filter(pl.col("is_view") == 1)
          .mode()
          .first()
          .alias("main_category_viewed")
    )
    # Материализация
    .collect(engine="streaming")
)

# Конвертация в Pandas
df_pd = session_df.to_pandas()

In [6]:
df_pd.head(3)

,user_session,user_id,weekday,is_weekend,start_hour,events_count,views,carts,purchases,made_purchase,added_to_cart,avg_price,max_price,main_category_viewed
0,7fc0221b-3213-434a-a5be-cc430f8ae296,518431894,7,1,16,1,1,0,0,0,0,1005.9300,1005.93,electronics
1,94ae8a9e-5bc4-4eba-9083-3d0fa2f55add,552253314,5,1,4,1,1,0,0,0,0,171.1800,171.18,None
2,8091d6fd-19f1-42b7-9326-28f2a51f1279,521518277,3,0,14,4,4,0,0,0,0,97.7775,252.21,None


БАЗОВЫЙ EDA

In [7]:
print(f"Сессий: {len(df_pd):,}")
print(f"Конверсия: {df_pd['made_purchase'].mean():.1%}")
print(f"Среднее views: {df_pd['views'].mean():.1f}")
print(f"Диапазон времени: {df_pd['start_hour'].min()}-{df_pd['start_hour'].max()}")

print("\nТОП-5 КАТЕГОРИЙ:")
print(df_pd['main_category_viewed'].value_counts().head())

Сессий: 4,156,287
Конверсия: 6.8%
Среднее views: 4.2
Диапазон времени: 0-23

ТОП-5 КАТЕГОРИЙ:
main_category_viewed
electronics    1911313
appliances      437272
computers       206012
apparel         178494
furniture       123087
Name: count, dtype: int64


ПРОВЕРКА КАЧЕСТВА ДАННЫХ

In [9]:
# Очистка дублей
print(f" До очистки: {len(df_pd):,} сессий")

df_clean = df_pd.drop_duplicates(subset=['user_session'], keep='first')
print(f" После очистки: {len(df_clean):,} сессий (-{len(df_pd)-len(df_clean)} дублей)")

# Проверка конверсии
print(f" Конверсия ДО:   {df_pd['made_purchase'].mean():.3%}")
print(f" Конверсия ПОСЛЕ: {df_clean['made_purchase'].mean():.3%}")

df_clean.to_csv('sessions_14days.csv', index=False)


 До очистки: 4,156,287 сессий
 После очистки: 4,156,113 сессий (-174 дублей)
 Конверсия ДО:   6.847%
 Конверсия ПОСЛЕ: 6.847%


In [ ]:
# Копируем файл на Google Drive
!cp /content/sessions_14days.csv /content/drive/MyDrive/sessions_14days.csv
print("Путь: MyDrive/sessions_14days.csv")

Путь: MyDrive/sessions_14days.csv
